## Quantile binning approach: EPO predictions

This notebook is for making dose-response function predictions given causalPFN discretized via quantiles.

In [1]:
import pandas as pd
import sys
sys.path.append("../../..")
import numpy as np
import torch
import datetime

from src.causalpfn.causal_estimator import CausalEstimator

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [3]:
## Main hyperparameter
N_DISC_VALUES = [3, 4]

In [ ]:
# Discretization function
def discretize_treatment(T: np.ndarray, N: int) -> np.ndarray:
    """Returns quantile method discretized version of T. Assumes range of T is [0, 1].

    Args:
        T (np.ndarray): The raw treatment data 
        N (int): The number of bins (equiv to the number of discrete treatment values)

    Returns:
        np.ndarray: The discretized treatment data
    """
    T_discrete = np.zeros(T.shape)
    bin_edges = np.percentile(T, np.linspace(0, 100, N + 1))
    for i in range(len(bin_edges) - 1):
        left_edge = bin_edges[i]
        right_edge = bin_edges[i + 1]
        ids = (T >= left_edge) & (T <= right_edge)
        avg = np.mean(T[ids])
        T_discrete[ids] = avg

    return T_discrete

In [5]:
## Synthetic data generation

data_name = "vahid-nonlinear"

np.random.seed(42)
n, d = 2000, 1
X = np.random.normal(2, 1, size=(n, d)).astype(np.float32)
T = (0.1 * X[:, 0] ** 2 - X[:, 0] + np.random.normal(1, 2, size=n)).astype(np.float32)
T = T - T.min() # Rescale
T = T / T.max() # Rescale
Y = (0.5 * T ** 2 - T * X[:, 0] + np.random.normal(0, 2, size=n)).astype(np.float32)
def drf(t): return 0.5 * t ** 2 - 2 * t # true dose-response funcion

df = pd.concat([
    pd.DataFrame(data=X, columns=["x"]), 
    pd.DataFrame(data=T, columns=["T"]), 
    pd.DataFrame(data=Y, columns=["Y"])
    ], axis=1)

In [ ]:
## Main inference loop
list_of_epos = [] # [(N_DISC, epos)], epos = [(mu_t0, mu_t1), (mu_t1, mu_t2), ... ]
for N_DISC in N_DISC_VALUES:
    print(f"N_DISC: {N_DISC}")
    T_discrete = discretize_treatment(T, N_DISC)
    discrete_treatment_levels = np.unique(T_discrete)
    epos = []
    for i, t in enumerate(discrete_treatment_levels[:-1]):
        # focus on two neighboring treatment levels and convert to binary T = 0, 1 values 
        t0, t1 = discrete_treatment_levels[i], discrete_treatment_levels[i + 1]
        ids = (np.abs(T_discrete - t0) < 1e-4) | (np.abs(T_discrete - t1) < 1e-4)
        T_temp = np.where(np.abs(T_discrete[ids] - t0) < 1e-4, 0, 1).astype(np.float32)
        X_temp = X[ids].astype(np.float32)
        Y_temp = Y[ids].astype(np.float32)
        # to predict cepo
        X_context = X_temp 
        t_context = T_temp
        y_context = Y_temp
        X_query = X_temp 
        t_all_ones = np.ones(X_query.shape[0], dtype=X_query.dtype)
        t_all_zeros = np.zeros(X_query.shape[0], dtype=X_query.dtype)
        causalpfn_cepo = CausalEstimator(
            device=device,
            verbose=True
        )
        causalpfn_cepo.fit(X_temp, T_temp, Y_temp)
        mu_vals = causalpfn_cepo._predict_cepo(
            X_context=X_context,
            t_context=t_context,
            y_context=y_context,
            X_query=np.concatenate([X_query, X_query], axis=0),
            t_query=np.concatenate([t_all_zeros, t_all_ones], axis=0),
            temperature=causalpfn_cepo.prediction_temperature,
        )
        mu_0 = (mu_vals[: X_query.shape[0]]).mean()
        mu_1 = (mu_vals[X_query.shape[0] :]).mean()
        epos.append((mu_0, mu_1))
    list_of_epos.append((N_DISC, epos))

N_DISC: 3
0: 0.28732237219810486
1: 0.4552024006843567
2: 0.6232631802558899


Predicting CEPO: 100%|██████████| 2666/2666 [00:05<00:00, 500.84it/s]


N_DISC: 4
0: 0.2593258023262024
1: 0.4050978422164917
2: 0.5055218935012817
3: 0.6511051058769226


Predicting CEPO: 100%|██████████| 2000/2000 [00:03<00:00, 656.70it/s]


In [17]:
## Create DataFrame and format it
# treatment_value_idx refers to which bin of the N_DISC the data is in. 
# E.g. N_DISC = 3 and treatment_value_idx = 1 (out of bins [0, 1, 2])
# refers to a treatment value of 0.5, and N_DISC = 4 and treatment_value_idx = 2
# refers to a treatment value of 2/3. 
multi_indices = pd.MultiIndex.from_tuples(
    [(N, i) for N in N_DISC_VALUES for i in range(N)],
    names=["N_DISC", "treatment_value_idx"]
)
cols = ["EPOs_1", "EPOs_2", "true_effect"]
data = []
for i, N in enumerate(N_DISC_VALUES):
    epos = list_of_epos[i][1]
    for j in range(N): 
        # iterate over treatment_value_idx values; first and last values 
        # have only one prediction
        if j == 0:
            epo_1 = np.nan
            epo_2 = epos[j][0]
        elif j == N - 1:
            epo_1 = epos[j - 1][1]
            epo_2 = np.nan
        else:
            epo_1 = epos[j - 1][1]
            epo_2 = epos[j][0]
        treatment_val = j / (N - 1)
        true_effect = drf(treatment_val)
        data.append([epo_1, epo_2, true_effect])
epo_df= pd.DataFrame(
    data=data,
    index=multi_indices,
    columns=cols
)

In [19]:
## Saving output
# For filenaming so as not to overwrite previous output 
month = datetime.datetime.now().month
date = datetime.datetime.now().day
hour = datetime.datetime.now().hour
minute = datetime.datetime.now().minute
date_string = f"{month}-{date}_{hour}h{minute}m"
# Save the DataFrame
file_name = "quantile_EPO_df_" + data_name + "_" + date_string
file_location = "../output"
epo_df.to_csv(f"{file_location}/{file_name}.csv")